# Foosball SAC Agent — Kaggle Training v2

Train a **Soft Actor-Critic (SAC)** reinforcement learning agent to play foosball using:
- **MuJoCo** physics simulation (`foosball_sim/v2/`)
- **Stable-Baselines3** SAC implementation
- **Protagonist-Antagonist** self-play curriculum via `SinglePlayerTrainingEngine`

**Repo:** https://github.com/carlkaziboni/Foosball_CU.git  
**Output:** models saved to `/kaggle/working/models/` — downloadable as zip at the end.

---
Run cells **top to bottom**. GPU (P100/T4) auto-detected and hyperparameters scaled accordingly.

## 1. Install System & Python Dependencies

Installs headless OpenGL libraries, MuJoCo, Stable-Baselines3, and all supporting packages.  
The repo is cloned (or pulled if already present) and added to `sys.path`.

In [ ]:
import subprocess, sys, os, shutil

def run(cmd):
    subprocess.check_call(cmd)

def pip(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *packages])

# ── System packages for headless OpenGL rendering ────────────────────────────
run(["apt-get", "install", "-y", "-q",
     "xvfb", "libgl1-mesa-glx", "libosmesa6", "libglfw3", "ffmpeg"])

# ── Python packages ────────────────────────────────────────────────────────────
pip(
    "mujoco",
    "stable-baselines3[extra]",
    "gymnasium",
    "shimmy>=0.2.0",
    "pyvirtualdisplay",
    "glfw",
    "tensorboard",
)

# ── Clone / update repo ────────────────────────────────────────────────────────
REPO_URL = "https://github.com/carlkaziboni/Foosball_CU.git"
REPO_DIR = "/kaggle/working/Foosball_CU"

# Always do a fresh shallow clone to guarantee we have the latest commit.
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
print(f"Repo cloned → {REPO_DIR}")

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Install any project-level requirements
req_file = os.path.join(REPO_DIR, "requirements.txt")
if os.path.exists(req_file):
    pip("-r", req_file)
    print("requirements.txt installed ✓")

print("\nCell 1 done ✓")

## 2. Headless Rendering Setup & GLFW Patch

Sets `MUJOCO_GL=osmesa`, starts a virtual display, then **patches `MujocoTableRenderMixin` to a no-op class** before `FoosballEnv` is ever imported.  
This prevents GLFW window errors in Kaggle's headless kernel.

In [ ]:
import os, sys, warnings

# ── Headless env vars — must be set BEFORE any mujoco/glfw import ─────────────
os.environ["MUJOCO_GL"]         = "osmesa"
os.environ["PYOPENGL_PLATFORM"] = "osmesa"

# ── Virtual display ────────────────────────────────────────────────────────────
from pyvirtualdisplay import Display
_display = Display(visible=False, size=(1280, 960))
_display.start()
os.environ["DISPLAY"] = ":0"
print("Virtual display started ✓")

# ── Silence gym deprecation warnings ──────────────────────────────────────────
warnings.filterwarnings("ignore", message=".*upgrade to Gymnasium.*")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="gym")

# ── Output directories ─────────────────────────────────────────────────────────
REPO_DIR  = "/kaggle/working/Foosball_CU"
MODEL_DIR = "/kaggle/working/models"
LOG_DIR   = "/kaggle/working/logs"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOG_DIR,   exist_ok=True)

# ── Patch GLFW mixin to no-op BEFORE FoosballEnv is imported ──────────────────
import ai_agents.v2.gym.mujoco_table_render_mixin as _mixin_module

class _HeadlessMixin:
    """No-op render mixin for Kaggle headless environment."""
    def __init__(self):
        self.viewer = None
        self.window = None
    def _initialize_glfw(self): pass
    def render(self, mode="human"): pass
    def close(self): pass

_mixin_module.MujocoTableRenderMixin = _HeadlessMixin
print("MujocoTableRenderMixin → HeadlessMixin ✓")

print(f"\nMUJOCO_GL  : {os.environ['MUJOCO_GL']}")
print(f"MODEL_DIR  : {MODEL_DIR}")
print(f"LOG_DIR    : {LOG_DIR}")
print("Cell 2 done ✓")

## 3. Environment Factory

Imports `FoosballEnv` (safe now that the mixin is patched) and defines the factory used everywhere.  
`observation_space` is **38-dim** continuous; `action_space` is **8-dim** continuous (4 rods × 2: linear + rotation).

In [ ]:
# Mixin is already patched — safe to import FoosballEnv now
from stable_baselines3.common.monitor import Monitor
from ai_agents.v2.gym.full_information_protagonist_antagonist_gym import FoosballEnv

# ── Global antagonist slot — set to a loaded SAC model to enable self-play ─────
_current_antagonist = None  # Phase 1: None = solo training


def sac_foosball_env_factory(x=None):
    """Creates a monitored FoosballEnv using the current global antagonist.
    In Phase 1, _current_antagonist is None (solo).
    In Phase 2, _current_antagonist is a frozen SAC model (self-play).
    """
    env = FoosballEnv(antagonist_model=_current_antagonist)
    env = Monitor(env, filename=os.path.join(LOG_DIR, "monitor"))
    return env


# ── Sanity check ───────────────────────────────────────────────────────────────
_test_env = sac_foosball_env_factory()
print("Observation space :", _test_env.observation_space)
print("Action space      :", _test_env.action_space)
_test_env.close()
del _test_env
print("Cell 3 done ✓  (antagonist slot ready)")


## 4. GPU Detection & Stable SAC Agent

Detects available hardware (CUDA P100/T4, MPS, or CPU) and defines `StableSACAgent` — a subclass of `SACFoosballAgent` with:
- `net_arch = [512, 512, 256]` (stable; avoids CPU/GPU tensor mismatch from large nets)
- `use_sde = False` (prevents NaN explosions on CUDA kernels)
- All other hyperparameters tuned for fast foosball learning

In [ ]:
import torch
from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import CheckpointCallback
from ai_agents.common.train.impl.sac_agent import SACFoosballAgent

# ── Detect device ──────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    print(f"Training device : CUDA — {gpu_name}")
elif torch.backends.mps.is_available():
    DEVICE = "mps"
    print("Training device : Apple Silicon MPS")
else:
    DEVICE = "cpu"
    print("Training device : CPU")

# ── StableSACAgent — subclass avoids recursion errors on notebook re-run ───────
_true_sac_init = SACFoosballAgent.__dict__["__init__"]

class StableSACAgent(SACFoosballAgent):
    """
    SAC agent continuing from models_new_longtrain weights.
    net_arch=[512,512,256], use_sde=False, gradient_steps=4.
    """

    def __init__(self, id, env=None,
                 log_dir=LOG_DIR,
                 model_dir=MODEL_DIR,
                 policy_kwargs=None):
        if policy_kwargs is None:
            policy_kwargs = dict(net_arch=[512, 512, 256])
        _true_sac_init(self, id=id, env=env,
                       log_dir=log_dir, model_dir=model_dir,
                       policy_kwargs=policy_kwargs)

    # ── Internal SAC factory ───────────────────────────────────────────────────
    def _make_sac(self, tensorboard_log=None):
        kwargs = dict(
            policy="MlpPolicy",
            env=self.env,
            policy_kwargs=self.policy_kwargs,
            device=DEVICE,
            learning_rate=3e-4,
            buffer_size=300_000,
            learning_starts=1_000,
            batch_size=256,
            tau=0.005,
            gamma=0.99,
            train_freq=4,
            gradient_steps=4,
            ent_coef="auto",
            target_entropy="auto",
            use_sde=False,
            verbose=1,
        )
        if tensorboard_log:
            kwargs["tensorboard_log"] = tensorboard_log
        return SAC(**kwargs)

    def initialize_agent(self):
        try:
            self.load()
            print(f"Agent {self.id} — loaded from checkpoint  ✓")
        except Exception as e:
            print(f"Agent {self.id} — no checkpoint ({e}), initialising fresh model.")
            self.model = self._make_sac()

    def learn(self, total_timesteps, extra_callbacks=None):
        if self.model is None:
            tb_log = os.path.join(LOG_DIR, f"sac_agent_{self.id}")
            self.model = self._make_sac(tensorboard_log=tb_log)

        callbacks = self.create_callback(self.env)
        if extra_callbacks:
            from stable_baselines3.common.callbacks import CallbackList
            callbacks = CallbackList([callbacks] + extra_callbacks)

        print(f"Agent {self.id} — training for {total_timesteps:,} timesteps ...")
        self.model.learn(
            total_timesteps=total_timesteps,
            callback=callbacks,
            tb_log_name=f"run_{self.id}",
            progress_bar=False,
            log_interval=1,
            reset_num_timesteps=False,
        )

    def create_callback(self, env):
        from stable_baselines3.common.callbacks import EvalCallback, CallbackList
        from ai_agents.common.train.impl.tensorboard_callback import (
            DetailedTensorboardCallback, FoosballMonitorCallback
        )
        # Best-model eval: run every 2 000 steps (was 3 000), 5 episodes
        eval_callback = EvalCallback(
            env,
            best_model_save_path=self.id_subdir + '/sac/best_model',
            log_path=self.log_dir,
            eval_freq=2_000,
            n_eval_episodes=5,
            render=False,
            deterministic=True,
            verbose=0,
        )
        tb_callback      = DetailedTensorboardCallback(verbose=0)
        monitor_callback = FoosballMonitorCallback(env, verbose=0)
        return CallbackList([eval_callback, tb_callback, monitor_callback])

print("StableSACAgent defined ✓")
print("Cell 4 done ✓")

## 5. Training Configuration

Continues from **`models_new_longtrain`** (1.5M timesteps pretrained).

| Setting | GPU | CPU |
|---|---|---|
| `total_epochs` | 150 | 30 |
| `epoch_timesteps` | 10 000 | 2 000 |
| **Total new timesteps** | **1 500 000** | **60 000** |
| `checkpoint_save_freq` | every epoch | every epoch |
| Timestep milestone saves | every 50 000 steps | every 10 000 steps |
| EvalCallback frequency | every 2 000 steps | every 2 000 steps |

**Checkpoint files produced per run:**
- `best_model/model.zip` — updated whenever EvalCallback finds a new best
- `epoch_N/model.zip` — saved every epoch (dense coverage)
- `timestep_XK/model.zip` — milestone saves every 50k steps
- `logs/training_rewards.csv` — per-epoch reward history for plotting

**Self-play curriculum (reward-triggered):**
Phase 1 = solo until mean reward ≥ `self_play_reward_threshold`.
Phase 2 = self-play, antagonist refreshed every `self_play_update_interval` epochs.

In [ ]:
IS_GPU = DEVICE == "cuda"

# ── Auto-scale based on available hardware ─────────────────────────────────────
if IS_GPU:
    total_epochs           = 150
    epoch_timesteps        = 10_000
    cycle_timesteps        = 1_000
    milestone_step_freq    = 50_000   # save timestep_XK every 50k steps on GPU
else:
    total_epochs           = 30
    epoch_timesteps        = 2_000
    cycle_timesteps        = 500
    milestone_step_freq    = 10_000   # save timestep_XK every 10k steps on CPU

# ── Override here if you want a custom run ─────────────────────────────────────
# total_epochs    = 20
# epoch_timesteps = 5_000

total_timesteps = total_epochs * epoch_timesteps

# ── Self-play curriculum ────────────────────────────────────────────────────────
self_play_reward_threshold = 500.0
self_play_reward_window    = 20
self_play_update_interval  = 5

# ── Pretrained starting point ──────────────────────────────────────────────────
# Set to None to start from scratch instead of continuing from longtrain.
PRETRAINED_SOURCE = "models_new_longtrain"   # folder name inside the cloned repo

print("=" * 54)
print(f"  Device              : {DEVICE.upper()}")
print(f"  Pretrained source   : {PRETRAINED_SOURCE or 'none (fresh start)'}")
print(f"  Epochs              : {total_epochs}")
print(f"  Timesteps / epoch   : {epoch_timesteps:,}")
print(f"  Total new timesteps : {total_timesteps:,}")
print(f"  Milestone save freq : every {milestone_step_freq:,} steps")
print(f"  Checkpoint / epoch  : every epoch")
print(f"  EvalCallback freq   : every 2 000 steps")
print(f"  Self-play threshold : mean reward ≥ {self_play_reward_threshold}")
print(f"  Model output dir    : {MODEL_DIR}")
print("=" * 54)
print("Cell 5 done ✓")

## 5.5. Seed Pretrained Weights

Copies `models_new_longtrain` from the cloned repo into `MODEL_DIR` so that  
`StableSACAgent.initialize_agent()` (Cell 6) loads these weights as the starting point.

Set `PRETRAINED_SOURCE = None` in Cell 5 to skip this and train from scratch.

In [ ]:
import shutil, os

PRETRAINED_DEST = os.path.join(MODEL_DIR, "0", "sac", "best_model")

if PRETRAINED_SOURCE:
    src_best = os.path.join(REPO_DIR, PRETRAINED_SOURCE, "0", "sac", "best_model", "model.zip")
    src_ckpt = os.path.join(REPO_DIR, PRETRAINED_SOURCE, "0", "sac", "best_model", "best_model.zip")

    if os.path.exists(src_best):
        os.makedirs(PRETRAINED_DEST, exist_ok=True)
        shutil.copy2(src_best, os.path.join(PRETRAINED_DEST, "model.zip"))
        print(f"Pretrained weights copied: {src_best}")
        print(f"  → {PRETRAINED_DEST}/model.zip")

        # Also copy best_model.zip if present (SACFoosballAgent.load fallback)
        if os.path.exists(src_ckpt):
            shutil.copy2(src_ckpt, os.path.join(PRETRAINED_DEST, "best_model.zip"))
            print(f"  → {PRETRAINED_DEST}/best_model.zip  (fallback copy)")

        # Record the source so we know where this run started from
        meta_path = os.path.join(MODEL_DIR, "pretrained_from.txt")
        with open(meta_path, "w") as f:
            f.write(f"source: {PRETRAINED_SOURCE}\n")
            f.write(f"source_path: {src_best}\n")
        print(f"  Provenance recorded → {meta_path}")
    else:
        print(f"⚠  Source weights not found at {src_best}")
        print("   Agent will initialise a fresh model in Cell 6.")
else:
    print("PRETRAINED_SOURCE=None — starting from scratch.")

print("\nCell 5.5 done ✓")

## 6. Agent Manager & Training Engine

`GenericAgentManager` owns one `StableSACAgent`.  
`SinglePlayerTrainingEngine` runs one epoch per call, saves checkpoints, and handles model reloading between epochs.

In [ ]:
from ai_agents.common.train.impl.generic_agent_manager import GenericAgentManager
from ai_agents.common.train.impl.single_player_training_engine import SinglePlayerTrainingEngine

# ── Agent manager ──────────────────────────────────────────────────────────────
agent_manager = GenericAgentManager(
    num_agents=1,
    environment_generator=sac_foosball_env_factory,
    agent_class=StableSACAgent,
)
agent_manager.initialize_training_agents()
agent_manager.initialize_frozen_best_models()

# NOTE: No engine is created here — Cell 7 creates a fresh SinglePlayerTrainingEngine
# each epoch so that FoosballEnv instances pick up the current _current_antagonist.

print("Agent manager  : 1 StableSACAgent ✓")
print("Cell 6 done ✓")


## 7. Run Training — Dense Checkpoints + Reward-Triggered Self-Play

**Checkpoints saved:**
- `best_model/` — whenever EvalCallback finds a new best (every 2 000 steps)
- `epoch_N/` — **every epoch** (full coverage of training history)
- `timestep_XK/` — every `milestone_step_freq` steps (50k on GPU, 10k on CPU)
- `logs/training_rewards.csv` — per-epoch reward log for offline plotting

**Continuing from:** `models_new_longtrain` (seeded in Cell 5.5).  
The agent's internal `num_timesteps` counter carries over from the pretrained model, so TensorBoard and SB3 logs correctly show the total training history.

In [ ]:
import time, csv, math
from stable_baselines3 import SAC

FROZEN_CHECKPOINT = os.path.join(MODEL_DIR, "0", "sac", "best_model", "model.zip")
MONITOR_CSV       = os.path.join(LOG_DIR, "monitor.monitor.csv")
REWARD_LOG_CSV    = os.path.join(LOG_DIR, "training_rewards.csv")

# ── Helpers ────────────────────────────────────────────────────────────────────
def read_recent_mean_reward(csv_path, window=20):
    if not os.path.exists(csv_path):
        return None
    rewards = []
    try:
        with open(csv_path, "r") as f:
            reader = csv.DictReader(row for row in f if not row.startswith("#"))
            for row in reader:
                rewards.append(float(row["r"]))
    except Exception:
        return None
    if not rewards:
        return None
    return sum(rewards[-window:]) / len(rewards[-window:])


def save_checkpoint(agent, path_suffix, total_ts):
    """Save model to MODEL_DIR/0/sac/<path_suffix>/model.zip"""
    ck_path = os.path.join(MODEL_DIR, "0", "sac", path_suffix)
    os.makedirs(ck_path, exist_ok=True)
    agent.model.save(os.path.join(ck_path, "model"))
    # Write a small metadata file alongside the checkpoint
    with open(os.path.join(ck_path, "info.txt"), "w") as f:
        f.write(f"total_timesteps_at_save: {total_ts}\n")
        f.write(f"path_suffix: {path_suffix}\n")


def log_epoch_reward(epoch, total_ts, mean_r, phase):
    """Append one row to the per-epoch reward CSV."""
    write_header = not os.path.exists(REWARD_LOG_CSV)
    with open(REWARD_LOG_CSV, "a", newline="") as f:
        w = csv.writer(f)
        if write_header:
            w.writerow(["epoch", "total_timesteps", "mean_reward", "phase"])
        w.writerow([epoch, total_ts, f"{mean_r:.3f}" if mean_r is not None else "n/a", phase])


# ── Training state ─────────────────────────────────────────────────────────────
wall_start = time.time()
global _current_antagonist
_current_antagonist = None

in_self_play     = False
self_play_epoch  = 0
epochs_completed = 0
total_ts_so_far  = 0

# Milestone tracking: next timestep threshold to save at
next_milestone_ts = milestone_step_freq

# If loading a pretrained model, offset milestones by its timestep count
try:
    _probe = SAC.load(FROZEN_CHECKPOINT, device="cpu")
    pretrained_ts = _probe.num_timesteps
    del _probe
    # Advance milestone pointer past already-trained steps
    next_milestone_ts = (math.floor(pretrained_ts / milestone_step_freq) + 1) * milestone_step_freq
    print(f"Pretrained model has {pretrained_ts:,} timesteps.")
    print(f"First milestone checkpoint at {next_milestone_ts:,} total timesteps.")
except Exception:
    pretrained_ts = 0
    print("No pretrained model found — starting milestone at 0.")

print(f"\nStarting training — {total_epochs} epochs  ({total_timesteps:,} new timesteps)\n")

for epoch in range(1, total_epochs + 1):

    # ── Refresh antagonist if in self-play ────────────────────────────────────
    if in_self_play and (self_play_epoch % self_play_update_interval == 0):
        try:
            _current_antagonist = SAC.load(FROZEN_CHECKPOINT, device=DEVICE)
            print(f"  [Epoch {epoch}] Antagonist refreshed ✓")
        except Exception as e:
            print(f"  [Epoch {epoch}] Antagonist reload failed ({e})")

    # ── Train one epoch ────────────────────────────────────────────────────────
    ep_engine = SinglePlayerTrainingEngine(
        agent_manager=agent_manager,
        environment_generator=sac_foosball_env_factory,
    )
    ep_engine.train(
        total_epochs=1,
        epoch_timesteps=epoch_timesteps,
        cycle_timesteps=cycle_timesteps,
    )
    epochs_completed  += 1
    total_ts_so_far   += epoch_timesteps
    if in_self_play:
        self_play_epoch += 1

    # ── Retrieve the trained agent so we can save checkpoints manually ────────
    protagonist_agent = agent_manager.get_training_agents()[0]

    # ── Dense epoch checkpoint (every epoch) ──────────────────────────────────
    save_checkpoint(protagonist_agent, f"epoch_{epoch:04d}", pretrained_ts + total_ts_so_far)
    print(f"  💾 epoch_{epoch:04d} checkpoint saved")

    # ── Milestone timestep checkpoint ─────────────────────────────────────────
    current_total_ts = pretrained_ts + total_ts_so_far
    while current_total_ts >= next_milestone_ts:
        label = f"timestep_{next_milestone_ts // 1000}K"
        save_checkpoint(protagonist_agent, label, current_total_ts)
        print(f"  🎯 {label} checkpoint saved  (total {current_total_ts:,} ts)")
        next_milestone_ts += milestone_step_freq

    # ── Read mean reward & log ─────────────────────────────────────────────────
    mean_r      = read_recent_mean_reward(MONITOR_CSV, window=self_play_reward_window)
    phase_label = "SELF-PLAY" if in_self_play else "SOLO"
    r_str       = f"{mean_r:.1f}" if mean_r is not None else "n/a"
    log_epoch_reward(epoch, current_total_ts, mean_r, phase_label)

    elapsed = time.time() - wall_start
    h, rem  = divmod(int(elapsed), 3600)
    m, s    = divmod(rem, 60)
    print(f"  [Epoch {epoch:>3}/{total_epochs}] {phase_label:<9}  "
          f"mean_r={r_str:<8}  total_ts={current_total_ts:,}  "
          f"elapsed={h:02d}h{m:02d}m{s:02d}s")

    # ── Self-play phase transition ─────────────────────────────────────────────
    if not in_self_play and mean_r is not None and mean_r >= self_play_reward_threshold:
        in_self_play    = True
        self_play_epoch = 0
        print(f"\n{'='*62}")
        print(f"  *** Reward threshold reached ({mean_r:.1f} ≥ {self_play_reward_threshold}) ***")
        print(f"  Switching to SELF-PLAY at epoch {epoch + 1}")
        print(f"{'='*62}\n")
        try:
            _current_antagonist = SAC.load(FROZEN_CHECKPOINT, device=DEVICE)
            print(f"  Initial antagonist loaded ✓")
        except Exception as e:
            print(f"  Could not load initial antagonist ({e})")

# ── Final cleanup ──────────────────────────────────────────────────────────────
_current_antagonist = None

elapsed = time.time() - wall_start
h, rem  = divmod(int(elapsed), 3600)
m, s    = divmod(rem, 60)
print("\n" + "=" * 62)
print(f"Training complete ✓  ({h:02d}h {m:02d}m {s:02d}s)")
print(f"New timesteps    : {total_timesteps:,}")
print(f"Total timesteps  : {pretrained_ts + total_timesteps:,}  (incl. pretrained)")
phase_str = f"triggered at reward ≥ {self_play_reward_threshold}" if in_self_play else f"never triggered (max r={r_str})"
print(f"Self-play        : {phase_str}")
print(f"Reward log       : {REWARD_LOG_CSV}")
print(f"Models saved to  : {MODEL_DIR}")
print("=" * 62)

## 8. Inspect Saved Checkpoints

Lists every file written under `MODEL_DIR` so you can verify checkpoints are present before downloading.

In [ ]:
import glob

print(f"Checkpoint inventory under {MODEL_DIR}:\n")

# Group by checkpoint folder
ck_dirs = sorted(set(
    os.path.dirname(f)
    for f in glob.glob(f"{MODEL_DIR}/**/*", recursive=True)
    if os.path.isfile(f)
))

for d in ck_dirs:
    files = [f for f in glob.glob(os.path.join(d, "*")) if os.path.isfile(f)]
    total_kb = sum(os.path.getsize(f) for f in files) / 1024
    label    = d.replace(MODEL_DIR + "/", "")
    # Read info.txt if present
    info_path = os.path.join(d, "info.txt")
    info_str  = ""
    if os.path.exists(info_path):
        with open(info_path) as f:
            info_str = "  " + f.read().strip().replace("\n", ", ")
    print(f"  {label:<40}  {total_kb:>7.0f} KB{info_str}")

# Print reward log summary if it exists
print()
if os.path.exists(REWARD_LOG_CSV):
    import csv as _csv
    rows = []
    with open(REWARD_LOG_CSV) as f:
        rows = list(_csv.DictReader(f))
    if rows:
        rewards = [float(r["mean_reward"]) for r in rows if r["mean_reward"] != "n/a"]
        print(f"Reward log  : {len(rows)} epochs recorded")
        if rewards:
            print(f"  best mean : {max(rewards):.1f}")
            print(f"  last mean : {rewards[-1]:.1f}")
else:
    print("(no reward log yet)")

print("\nCell 8 done ✓")

## 9. Evaluate Trained Agent

Loads the best saved model and runs `NUM_EVAL_EPISODES` deterministic episodes.  
Prints per-episode reward and a summary (mean / min / max).

In [ ]:
NUM_EVAL_EPISODES = 10

# Re-initialise frozen models so they pick up the latest saved checkpoint
agent_manager.initialize_frozen_best_models()
protagonist = agent_manager.get_frozen_best_models()[0]

eval_env = sac_foosball_env_factory()
episode_rewards = []

print(f"Evaluating trained agent — {NUM_EVAL_EPISODES} episodes (deterministic)\n")

for ep in range(NUM_EVAL_EPISODES):
    obs, _ = eval_env.reset()
    done = False
    total_reward = 0.0
    steps = 0

    while not done:
        # protagonist is a SACFoosballAgent — its predict() already unwraps (action, state)
        # and returns just the action array directly
        action = protagonist.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        total_reward += reward
        steps += 1
        done = terminated or truncated

    episode_rewards.append(total_reward)
    print(f"  Episode {ep + 1:>2}/{NUM_EVAL_EPISODES}  reward: {total_reward:>8.2f}  steps: {steps}")

eval_env.close()

mean_r = sum(episode_rewards) / len(episode_rewards)
print(f"\n{'─' * 40}")
print(f"  Mean reward : {mean_r:.2f}")
print(f"  Min  reward : {min(episode_rewards):.2f}")
print(f"  Max  reward : {max(episode_rewards):.2f}")
print(f"{'─' * 40}")
print("Cell 9 done ✓")


## 9.5. Plot Training Reward Curve

Reads `logs/training_rewards.csv` (written every epoch during Cell 7) and plots  
the mean reward progression over the full run.

In [ ]:
import csv, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

if not os.path.exists(REWARD_LOG_CSV):
    print("No reward log found — run Cell 7 first.")
else:
    rows = []
    with open(REWARD_LOG_CSV) as f:
        rows = [r for r in csv.DictReader(f) if r["mean_reward"] != "n/a"]

    if not rows:
        print("Reward log is empty.")
    else:
        epochs   = [int(r["epoch"])          for r in rows]
        ts       = [int(r["total_timesteps"]) for r in rows]
        rewards  = [float(r["mean_reward"])   for r in rows]
        phases   = [r["phase"]                for r in rows]

        # Rolling mean (window=10)
        def roll(arr, w=10):
            return [np.mean(arr[max(0,i-w+1):i+1]) for i in range(len(arr))]

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        fig.suptitle("Training Reward Progression", fontsize=13, fontweight="bold")

        for ax, x_vals, x_label in [
            (axes[0], epochs, "Epoch"),
            (axes[1], ts,     "Total Timesteps"),
        ]:
            # Shade solo vs self-play regions
            in_sp = False
            for i, (xv, ph) in enumerate(zip(x_vals, phases)):
                if ph == "SELF-PLAY" and not in_sp:
                    ax.axvline(xv, color="purple", ls="--", lw=1.2, alpha=0.7,
                               label="Self-play starts")
                    in_sp = True

            ax.plot(x_vals, rewards, "o", ms=3, alpha=0.35, color="#4C9BE8")
            ax.plot(x_vals, roll(rewards), "-", lw=2.2, color="#1a6bbf",
                    label="Rolling mean (w=10)")
            ax.set_xlabel(x_label)
            ax.set_ylabel("Mean Episode Reward")
            ax.set_title(f"Reward vs {x_label}")
            ax.grid(alpha=0.3)
            ax.legend(fontsize=9)

        plt.tight_layout()
        plot_path = os.path.join(LOG_DIR, "training_reward_curve.png")
        plt.savefig(plot_path, dpi=150, bbox_inches="tight")
        print(f"Plot saved → {plot_path}")

        print(f"\nEpochs logged  : {len(rows)}")
        print(f"Best mean r    : {max(rewards):.1f}  (epoch {epochs[np.argmax(rewards)]})")
        print(f"Last mean r    : {rewards[-1]:.1f}")
        sp_epochs = sum(1 for p in phases if p == "SELF-PLAY")
        print(f"Self-play eps  : {sp_epochs}/{len(rows)}")
print("Cell 9.5 done ✓")

## 10. Package & Download Models

Zips `models/` into `/kaggle/working/foosball_sac_models.zip`.  
The archive is visible in the Kaggle **Output** tab — click the file to download.

In [ ]:
import shutil, glob

ARCHIVE_PATH = "/kaggle/working/foosball_sac_models"

shutil.make_archive(
    base_name=ARCHIVE_PATH,
    format="zip",
    root_dir="/kaggle/working",
    base_dir="models",
)

zip_path = f"{ARCHIVE_PATH}.zip"
zip_size_mb = os.path.getsize(zip_path) / 1e6

print(f"Archive created  : {zip_path}")
print(f"Archive size     : {zip_size_mb:.2f} MB")
print(f"\nContents:")
for f in sorted(glob.glob(f"{MODEL_DIR}/**/*", recursive=True)):
    if os.path.isfile(f):
        kb = os.path.getsize(f) / 1024
        print(f"  {f}  ({kb:.1f} KB)")

print("\nDownload from the Kaggle Output tab.")
print("Cell 10 done ✓")